# Internal Variability in Climate Projections
<br>

**Internal variability** is the third source of uncertainty in climate projections, alongside model uncertainty and scenario uncertainty. It arises from the inherently chaotic behaviour of the climate system itself — interactions among the atmosphere, ocean, land surface, and sea ice produce natural fluctuations that are not predictable beyond weather timescales.

Internal variability is distinct from *external* natural forcing (volcanic eruptions, solar fluctuations) because it originates *inside* the climate system. It is most apparent at **regional scales** and **short time horizons** (seasonal to decadal). At multi-decadal scales, model and scenario uncertainty progressively dominate.

**Intended Application:** As a user, I want to <span style="color:#FF0000">**understand and quantify internal variability**</span> in climate projections:
- Visualize within-model spread across CMIP6 ensemble members for extreme precipitation
- Compare internal variability to inter-model spread for the same variable
- Understand when internal variability dominates and why it demands special data handling
- Apply a data-pooling strategy to robustly estimate statistics under high internal variability

> **For a deeper scientific treatment**, see the [IPCC AR5 uncertainty guidance](https://www.ipcc.ch/site/assets/uploads/2017/08/AR5_Uncertainty_Guidance_Note.pdf), [Hawkins & Sutton (2009)](https://journals.ametsoc.org/view/journals/bams/90/8/2009bams2607_1.xml), and [climatewest.ca: Uncertainty 101](https://climatewest.ca/2022/09/27/uncertainty-101-understanding-climate-models/).

> **Note:** This notebook is part of a series. For inter-model spread see `model_uncertainty.ipynb`; for a full decomposition across all three uncertainty sources see `uncertainty_decomposition.ipynb`.

**Runtime:** Approximately **13–18 minutes** with default settings.

### Step 0: Setup

In [ ]:
from climakitae.explore.uncertainty import _area_wgt_average
import climakitae as ck
from uncertainty import new_core_get_ensemble_data
from climakitae.util.colormap import read_ae_colormap
from climakitae.new_core.user_interface import ClimateData

import xarray as xr
import numpy as np
import warnings
from scipy.stats import gaussian_kde
from dask.diagnostics import ProgressBar

import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import matplotlib.patches as mpatches
from bokeh.models import HoverTool

import holoviews as hv
hv.extension('bokeh')  # Load the bokeh backend for HoloViews
import panel as pn
pn.extension()

warnings.filterwarnings("ignore")
%config InlineBackend.figure_format = 'svg' # Make plots look better in the notebook environment.

%load_ext autoreload
%autoreload 2

### Step 1: Retrieve data

#### Step 1a: Downscaled WRF precipitation (Cal-Adapt AE models)

We use **monthly precipitation** because precipitation has among the highest internal variability of any climate variable — making it the clearest case for illustrating why internal variability matters and how to handle it.

Set the global warming level of interest. This controls the future period used throughout.

In [2]:
cd = ClimateData()
cd.catalog('cadcat')
cd.variable('prec')
cd.table_id('mon')
cd.activity_id('WRF')
cd.institution_id('UCLA')
cd.experiment_id(['historical', 'ssp370'])
cd.grid_label('d02')
cd.processes({'filter_unadjusted_models': 'no'})

wrf_ds = cd.get()
# Filter to your 8 models and clip near-zero values
wrf_ds = wrf_ds.clip(0.1)

2026-06-15 14:10:02 - climakitae.new_core.user_interface - INFO - Initializing ClimateData interface
2026-06-15 14:10:02 - climakitae.new_core.dataset_factory - INFO - DatasetFactory initialized with 3 validators and 12 processors
2026-06-15 14:10:02 - climakitae.new_core.user_interface - INFO - ClimateData initialization successful
2026-06-15 14:10:02 - climakitae.new_core.user_interface - INFO - ✅ Ready to query!
2026-06-15 14:10:02 - climakitae.new_core.user_interface - INFO - Catalog set to: cadcat
2026-06-15 14:10:02 - climakitae.new_core.user_interface - INFO - Variable set to: prec
2026-06-15 14:10:02 - climakitae.new_core.user_interface - INFO - Table ID set to: mon
2026-06-15 14:10:02 - climakitae.new_core.user_interface - INFO - Activity ID set to: WRF
2026-06-15 14:10:02 - climakitae.new_core.user_interface - INFO - Institution ID set to: UCLA
2026-06-15 14:10:02 - climakitae.new_core.user_interface - INFO - Experiment ID(s) set to: ['historical', 'ssp370']
2026-06-15 14:10:

#### Step 1b: CMIP6 ensemble members for the same models

For each of the eight Cal-Adapt models, CMIP6 provides multiple **ensemble members** — runs of the same model with slightly different initial conditions. The spread across ensemble members captures internal variability, since the model structure and forcing are identical.

In [3]:
warm_level = 3.0  # set your target

# ── Historical slice (unchanged logic) ───────────────────────────────────────
hist_wrf = wrf_ds.sel(time=slice('1981', '2010'))

# ── Future: warming-level slice via new_core (replaces get_warm_level loop) ──
cd_ssp = ClimateData()
cd_ssp.catalog('cadcat')
cd_ssp.variable('prec')
cd_ssp.table_id('mon')
cd_ssp.activity_id('WRF')
cd_ssp.institution_id('UCLA')
cd_ssp.experiment_id(['historical', 'ssp370'])
cd_ssp.grid_label('d02')
cd_ssp.processes({'filter_unadjusted_models': 'no'})
cd_ssp.processes({'warming_levels': warm_level})
ssp_wrf = cd_ssp.get()
ssp_wrf

2026-06-15 14:10:06 - climakitae.new_core.user_interface - INFO - Initializing ClimateData interface
2026-06-15 14:10:06 - climakitae.new_core.dataset_factory - INFO - DatasetFactory initialized with 3 validators and 12 processors
2026-06-15 14:10:06 - climakitae.new_core.user_interface - INFO - ClimateData initialization successful
2026-06-15 14:10:06 - climakitae.new_core.user_interface - INFO - ✅ Ready to query!
2026-06-15 14:10:06 - climakitae.new_core.user_interface - INFO - Catalog set to: cadcat
2026-06-15 14:10:06 - climakitae.new_core.user_interface - INFO - Variable set to: prec
2026-06-15 14:10:06 - climakitae.new_core.user_interface - INFO - Table ID set to: mon
2026-06-15 14:10:06 - climakitae.new_core.user_interface - INFO - Activity ID set to: WRF
2026-06-15 14:10:06 - climakitae.new_core.user_interface - INFO - Institution ID set to: UCLA
2026-06-15 14:10:06 - climakitae.new_core.user_interface - INFO - Experiment ID(s) set to: ['historical', 'ssp370']
2026-06-15 14:10:

<xarray.Dataset> Size: 3GB
Dimensions:            (sim: 5, time: 1440, y: 340, x: 270)
Coordinates:
  * sim                (sim) object 40B 'wrf_ucla_ec-earth3-veg_ssp370_r1i1p1...
  * time               (time) datetime64[ns] 12kB 1980-09-01 ... 2100-08-01
  * y                  (y) float64 3kB -3.341e+05 -3.251e+05 ... 2.717e+06
  * x                  (x) float64 2kB -4.728e+06 -4.719e+06 ... -2.307e+06
    lakemask           (y, x) float32 367kB dask.array<chunksize=(330, 263), meta=np.ndarray>
    landmask           (y, x) float32 367kB dask.array<chunksize=(330, 263), meta=np.ndarray>
    lat                (y, x) float32 367kB dask.array<chunksize=(330, 263), meta=np.ndarray>
    lon                (y, x) float32 367kB dask.array<chunksize=(330, 263), meta=np.ndarray>
    Lambert_Conformal  int32 4B ...
Data variables:
    prec               (sim, time, y, x) float32 3GB dask.array<chunksize=(1, 386, 262, 208), meta=np.ndarray>
Attributes: (181)

In [ ]:
# Filter both to the same simulations that reached the warming level
common_sims = [
    s for s in ssp_wrf.sim.values if s in hist_wrf.sim.values]
ssp_wrf = ssp_wrf.sel(sim=common_sims)
hist_wrf = hist_wrf.sel(sim=common_sims)

ssp_wrf = ck.load(ssp_wrf.unify_chunks(), progress_bar=True)
hist_wrf = ck.load(hist_wrf.unify_chunks(), progress_bar=True)
wrf_ds = ck.load(wrf_ds.unify_chunks(), progress_bar=True)

# ── Percentiles and delta (unchanged logic) ───────────────────────────────────
cads_hist_percentile = hist_wrf.chunk(
    {'time': -1}).quantile([.99], dim='time').compute().squeeze()
cads_ssp_percentile = ssp_wrf.chunk(
    {'time': -1}).quantile([.99], dim='time').compute().squeeze()
cads_delta_percentile = (cads_ssp_percentile - cads_hist_percentile).compute()

Processing data to read 2.46 GB of data into memory... 
[########################################] | 100% Completed | 543.70 s
Complete!
Processing data to read 631.75 MB of data into memory... 
[########################################] | 100% Completed | 146.01 s
Complete!
Available memory: 3.10 GB
Total memory of input data: 3.94 GB


In [ ]:
cmip_names = [
    'EC-Earth3', 'EC-Earth3-Veg', 'CESM2', 'CNRM-ESM2-1', 'FGOALS-g3',
    'MIROC6', 'TaiESM1', 'MPI-ESM1-2-HR'
]

hist_cae_ds, warm_cae_ds = new_core_get_ensemble_data(
    variable='pr',
    # your existing DataParameters object, still needed for spatial clipping
    selections=selections,
    cmip_names=cmip_names,
    warm_level=warm_level,
)

In [ ]:
hist_cae_ds.member_id

In [ ]:
hist_cae_ds.simulation

In [ ]:
hist_cae_ds = ck.load(hist_cae_ds, progress_bar=True)
warm_cae_ds = ck.load(warm_cae_ds, progress_bar=True)

In [ ]:
# 99th percentile per ensemble member, historical and future
hist_cae_p99 = hist_cae_ds.quantile(0.99, dim="time").squeeze()
warm_cae_p99 = warm_cae_ds.quantile(0.99, dim="time").squeeze()
delta_cae_p99 = warm_cae_p99 - hist_cae_p99

### Step 2: Visualize internal variability

#### Step 2a: Within-model ensemble spread — spatial maps

Each row below shows all ensemble members for one model. Because the model structure and forcing are identical across members, differences between maps within a row are **internal variability** alone.

> Compare rows across models to see inter-model spread. Compare maps within a row to see internal variability.

In [ ]:
# Shared map helper
def make_precip_map(data, title, vmin, vmax, diverging=False,
                    width=220, height=220):
    """Single precipitation map using holoviews QuadMesh."""
    cmap = read_ae_colormap(
        cmap="ae_diverging_r" if diverging else "ae_blue", cmap_hex=True
    )
    hover = HoverTool(tooltips=[
        ("Lon (°E)", "@x"), ("Lat (°N)", "@y"), ("Precip (mm/mo)", "@z")
    ])
    return hv.QuadMesh((data["lon"], data["lat"], data)).opts(
        tools=[hover], colorbar=True, cmap=cmap,
        symmetric=diverging, clim=(vmin, vmax),
        xaxis=None, yaxis=None,
        clabel="mm / month",
        title=title, width=width, height=height,
    )

In [ ]:
# Plot change in 99th-percentile precipitation per ensemble member, grouped by model
# Rename spatial dims to lon/lat for the map helper
delta_plot_ds = delta_cae_p99.rename({"x": "lon", "y": "lat"})
all_panels = None
for sim in np.unique(delta_plot_ds.simulation.values):
    sim_data = delta_plot_ds.where(delta_plot_ds.simulation == sim, drop=True)
    for mid in range(len(sim_data.member_id.values)):
        panel = make_precip_map(
            data=sim_data.pr.drop("simulation").isel(member_id=mid),
            title=f"{sim}  member {mid + 1}",
            vmin=-300, vmax=300,
            diverging=True,
        )
        all_panels = panel if all_panels is None else all_panels + panel

(
    all_panels.cols(6)
    .opts(
        hv.opts.Layout(
            merge_tools=True,
            toolbar="below",
            title=f"Change in 99th-percentile monthly precipitation at {warm_level}°C warming — per ensemble member"
        )
    )
)

Maps within each model row share identical forcing — differences between them are **pure internal variability**. Notice that some models show both strongly positive and strongly negative changes across their ensemble members, indicating that the internal variability signal can be as large as or larger than the forced response.

#### Step 2b: Internal variability vs. inter-model spread — range chart

The chart below makes the comparison between internal variability and model uncertainty directly quantitative. For each model, the **vertical bar** shows the range of 99th-percentile precipitation changes across ensemble members (internal variability). The **yellow dot** shows where the single downscaled WRF member for that model lands within that range.

First we compute area-averaged statistics for a sub-region of interest.

In [ ]:
# Define sub-region — modify to your area of interest
lat0, lat1 = 36.0, 39.0
lon0, lon1 = -123.0, -119.5

In [ ]:
wrf_ds['prec'] = wrf_ds['prec'].clip(0.1)

In [ ]:
# ── Spatial subset using lat/lon masks ───────────────────────────────────────
lat_mask = (wrf_da.lat >= lat0) & (wrf_da.lat <= lat1)
lon_mask = (wrf_da.lon >= lon0) & (wrf_da.lon <= lon1)
region_mask = lat_mask & lon_mask

reg_wrf_da = wrf_da.where(region_mask, drop=True)
reg_hist_wrf = reg_wrf_da.sel(time=slice("1981", "2010"))

reg_ssp_wrf = ssp_wrf["prec"].where(region_mask, drop=True).sel(
    sim=[s for s in reg_wrf_da.sim.values if s in ssp_wrf.sim.values]
)

print("Shape after masking:", reg_wrf_da.shape)
print("y size:", reg_wrf_da.y.size)
print("x size:", reg_wrf_da.x.size)

# ── 99th percentile and relative change ──────────────────────────────────────
reg_hist_p99 = (
    reg_hist_wrf
    .chunk({"time": -1})
    .quantile(0.99, dim="time")
    .compute()
    .squeeze()
)
reg_ssp_p99 = (
    reg_ssp_wrf
    .chunk({"time": -1})
    .quantile(0.99, dim="time")
    .compute()
    .squeeze()
)

# Relative change: (future - present) / present * 100
reg_wrf_delta = (((reg_ssp_p99 - reg_hist_p99) / reg_hist_p99) * 100).compute()

In [ ]:
# Area-average CMIP6 ensemble spread over the same sub-region
cae_rel_delta = (delta_cae_p99 / hist_cae_p99 * 100).compute()
reg_cae_delta = _area_wgt_average(
    cae_rel_delta.sel(y=slice(lat0, lat1), x=slice(lon0, lon1))
)

In [ ]:
sims = np.unique(reg_cae_delta.simulation.values)
bar_tops, bar_floors, medians = [], [], []

for sim in sims:
    mask = reg_cae_delta.simulation == sim
    ens = reg_cae_delta.pr.isel(member_id=mask)
    if ens.size == 0:
        print(f"⚠️  No data for {sim} — skipping")
        continue
    bar_tops.append(float(ens.max(skipna=True)))
    bar_floors.append(float(ens.min(skipna=True)))
    medians.append(float(ens.median(skipna=True)))

In [ ]:
# Relative change: (future - present) / present * 100, then spatial mean
reg_wrf_delta = (((reg_ssp_p99 - reg_hist_p99) / reg_hist_p99)
                 * 100).mean(["y", "x"]).compute()

In [ ]:
# ── Area-average CMIP6 ensemble over sub-region ───────────────────────────────
cae_rel_delta = (delta_cae_p99 / hist_cae_p99 * 100).compute()

reg_cae_delta = cae_rel_delta.sel(
    y=slice(lat0, lat1),
    x=slice(lon0, lon1)
)

# _area_wgt_average uses ds.y for weights — works directly here
# since y IS latitude in degrees for this regridded CMIP6 data
reg_cae_delta = _area_wgt_average(reg_cae_delta)

# ── Build per-model stats ─────────────────────────────────────────────────────
sims = np.unique(reg_cae_delta.simulation.values)
bar_tops, bar_floors, medians, wrf_vals = [], [], [], []

for sim in sims:
    mask = reg_cae_delta.simulation == sim
    ens = reg_cae_delta.pr.isel(member_id=mask)
    bar_tops.append(float(ens.max(skipna=True)))
    bar_floors.append(float(ens.min(skipna=True)))
    medians.append(float(ens.median(skipna=True)))

    wrf_sim = wrf_lookup.get(sim)
    if wrf_sim is None:
        wrf_vals.append(float("nan"))
    else:
        subset = reg_wrf_delta.where(
            reg_wrf_delta.sim == wrf_sim, drop=True
        ).squeeze()
        wrf_vals.append(float(subset) if subset.size > 0 else float("nan"))

# ── Plot ──────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 5))

ax.vlines(
    x=sims, ymax=bar_tops, ymin=bar_floors,
    linewidths=22, color="#4472A8", alpha=0.35,
    label="CMIP6 ensemble range\n(internal variability)",
)
ax.scatter(
    sims, medians,
    zorder=3, color="#4472A8", s=600, marker="_", linewidths=2.5,
    label="CMIP6 ensemble median",
)
ax.scatter(
    sims, wrf_vals,
    zorder=4, edgecolors="#1a1a1a", facecolors="#C8A83A",
    s=160, label="Downscaled WRF member",
)
ax.axhline(0, color="#aaaaaa", lw=0.8, ls="dashed")
ax.set_ylabel("% change in 99th-percentile monthly precipitation", fontsize=11)
ax.set_xlabel("Model", fontsize=11)
ax.set_title(
    f"Internal variability vs. inter-model spread — {warm_level}°C warming (SSP3-7.0)\n"
    "Sub-region average",
    fontsize=11, pad=10,
)
ax.legend(framealpha=0.9, fontsize=9,
          bbox_to_anchor=(1.02, 0.9), loc="upper left")
ax.spines[["top", "right"]].set_visible(False)
ax.tick_params(axis="x", rotation=25, labelsize=9)
ax.tick_params(axis="y", labelsize=9)
plt.tight_layout()
plt.show()

**Key observation:** The blue bars (internal variability within each model) are typically **larger** than the spread between model medians. This means that for extreme precipitation, choosing a different starting condition within the same model produces more spread than choosing a completely different model. This is the defining characteristic of a high-internal-variability regime.

> **Implication:** Because the single downscaled WRF member (yellow dot) is just one draw from the blue distribution, conclusions drawn from a single-member projection are highly sensitive to which draw was made — not to fundamental differences in model physics.

#### Step 2c: Ridgeline — distribution shape per model

The range chart shows the min–max; the ridgeline shows the **full distribution shape** across ensemble members for each model. Wider, flatter ridges = higher internal variability within that model.

In [ ]:
MODEL_COLORS = [
    "#4472A8", "#C0664A", "#5A9E6F", "#8B67B5",
    "#C8A83A", "#3D9BAA", "#B84D7A", "#6B7EC2",
]

offset_step = 1.2
n = len(sims)
fig, ax = plt.subplots(figsize=(9, 0.9 * n + 1.5))

for idx, (sim, color) in enumerate(zip(sims, MODEL_COLORS)):
    mask = reg_cae_delta.simulation == sim
    ens = reg_cae_delta.pr.isel(member_id=mask)
    vals = ens.values.flatten()
    vals = vals[~np.isnan(vals)]
    if len(vals) < 3:
        continue

    kde = gaussian_kde(vals, bw_method=0.5)
    x_grid = np.linspace(vals.min() - 5, vals.max() + 5, 300)
    y_kde = kde(x_grid)
    y_kde = y_kde / y_kde.max()
    base = idx * offset_step

    ax.fill_between(x_grid, base, base + y_kde,
                    color=color, alpha=0.35, zorder=n - idx)
    ax.plot(x_grid, base + y_kde, color=color, lw=1.5, zorder=n - idx)

    # Median
    med = float(np.median(vals))
    med_y = kde(med)[0] / y_kde.max()
    ax.plot([med, med], [base, base + med_y],
            color=color, lw=1.2, ls="dashed", alpha=0.85, zorder=n - idx + 1)

    # WRF single member — use wrf_lookup to map CMIP name → WRF sim name
    wrf_sim = wrf_lookup.get(sim)
    if wrf_sim is not None:
        subset = reg_wrf_delta.where(
            reg_wrf_delta.sim == wrf_sim, drop=True
        ).squeeze()
        if subset.size > 0:
            wrf_val = float(subset)
            wrf_y = kde(wrf_val)[0] / y_kde.max()
            ax.scatter([wrf_val], [base + wrf_y], color="#C8A83A",
                       edgecolors="#1a1a1a", s=60, zorder=n, linewidths=0.8)

    ax.text(-0.52, base + 0.08, sim,
            ha="right", va="bottom", fontsize=9,
            color=color, fontweight="medium",
            transform=ax.get_yaxis_transform())

# Legend
legend_elements = [
    mpatches.Patch(color="#4472A8", alpha=0.4, label="Ensemble distribution"),
    Line2D([0], [0], ls="dashed", color="#555",
           lw=1.2, label="Ensemble median"),
    Line2D([0], [0], marker="o", color="w", markerfacecolor="#C8A83A",
           markeredgecolor="#1a1a1a", markersize=7, label="WRF single member"),
]
ax.legend(handles=legend_elements, fontsize=9,
          loc="lower right", framealpha=0.9)
ax.axvline(0, color="#aaaaaa", lw=0.8, ls="dashed")
ax.set_xlabel("% change in 99th-percentile monthly precipitation", fontsize=11)
ax.set_title(
    f"Within-model distribution of ensemble members — {warm_level}°C warming (SSP3-7.0)",
    fontsize=11, pad=10,
)
ax.set_yticks([])
ax.spines[["top", "right", "left"]].set_visible(False)
ax.tick_params(labelsize=10)
ax.set_ylim(-0.3, n * offset_step + 0.5)
plt.tight_layout()
plt.show()

Wide, flat distributions indicate models with high internal variability — the single WRF member (yellow dot) could have landed anywhere in that distribution. Narrow, peaked distributions indicate models where ensemble members cluster tightly, meaning the forced response is more consistently captured.

### Step 3: Handling high internal variability — data pooling

When internal variability is large relative to the forced signal (as shown above for extreme precipitation), computing statistics from a single ensemble member per model gives unreliable results — the estimate depends heavily on which member was chosen, not on the underlying climate physics.

The recommended approach is to **pool all ensemble members across all models** before computing statistics. This treats each member as an equally plausible realisation of the climate, increasing the effective sample size by up to a factor of eight.

> **When is pooling appropriate?**  
> - Internal variability dominates model uncertainty (shown in Step 2b)  
> - The variable of interest has a small effective sample size (rare extremes)  
> - You can assume all models plausibly represent the relevant physical processes

> **When not to pool:** If inter-model spread dominates (e.g. end-of-century temperature), keeping models separate preserves the scientifically meaningful differences between them. See `uncertainty_decomposition.ipynb`.

In [ ]:
box_ssp_wrf = ssp_wrf["prec"]

# Filter both to the same models
common_sims = [s for s in box_ssp_wrf.sim.values
               if s in box_hist_wrf.sim.values]
box_ssp_wrf = box_ssp_wrf.sel(sim=common_sims)
box_hist_wrf = box_hist_wrf.sel(sim=common_sims)

In [ ]:
box_hist_wrf = ck.load(box_hist_wrf.unify_chunks(), progress_bar=True)

In [ ]:
# Per-model mean approach
box_hist_p99_mmm = box_hist_wrf.chunk(
    dict(time=-1)).quantile(0.99, dim="time").compute().squeeze()
box_ssp_p99_mmm = box_ssp_wrf.chunk(
    dict(time=-1)).quantile(0.99, dim="time").compute().squeeze()

hist_mmm = box_hist_p99_mmm.mean(dim="sim").compute().squeeze()
ssp_mmm = box_ssp_p99_mmm.mean(dim="sim").compute().squeeze()
delta_mmm = (ssp_mmm - hist_mmm).compute().squeeze()

# Pooled approach — stack all simulations and time into one dimension
hist_pool = box_hist_wrf.stack(index=["sim", "time"]).compute()
ssp_pool = box_ssp_wrf.stack(index=["sim", "time"]).compute()

hist_pool_p99 = hist_pool.chunk(
    dict(index=-1)).quantile(0.99, dim="index").compute().squeeze()
ssp_pool_p99 = ssp_pool.chunk(
    dict(index=-1)).quantile(0.99, dim="index").compute().squeeze()
delta_pool = (ssp_pool_p99 - hist_pool_p99).compute()

In [ ]:
def get_ks_pval_df(
    sample1: xr.Dataset | xr.DataArray,
    sample2: xr.Dataset | xr.DataArray,
    sig_lvl: float = 0.05,
) -> pd.DataFrame:
    """Two-sample KS test at every spatial point; returns significant grid cells."""
    # Extract DataArray if Dataset passed in
    if isinstance(sample1, xr.Dataset):
        sample1 = sample1[list(sample1.data_vars)[0]]
    if isinstance(sample2, xr.Dataset):
        sample2 = sample2[list(sample2.data_vars)[0]]

    sample1 = sample1.stack(allpoints=["y", "x"]).squeeze()
    sample2 = sample2.stack(allpoints=["y", "x"]).squeeze()
    sample1, sample2 = xr.align(sample1, sample2, join="inner")

    non_spatial_dims = [d for d in sample1.dims if d != "allpoints"]
    if len(non_spatial_dims) != 1:
        raise ValueError(
            f"Expected exactly one non-spatial dim after stacking, got {non_spatial_dims}."
        )
    core_dim = non_spatial_dims[0]

    def ks_stat_2sample(s1, s2):
        try:
            d_statistic, p_value = stats.kstest(s1, s2)
        except (ValueError, ZeroDivisionError):
            d_statistic, p_value = np.nan, np.nan
        return d_statistic, p_value

    _, p_value = xr.apply_ufunc(
        ks_stat_2sample,
        sample1,
        sample2,
        input_core_dims=[[core_dim], [core_dim]],
        exclude_dims=set((core_dim,)),
        output_core_dims=[[], []],
        vectorize=True,
    )

    p_df = p_value.dropna(dim="allpoints")
    p_df = p_value.rename("p_value")
    p_df = p_df.unstack("allpoints").to_dataframe().reset_index()
    p_df = p_df[["lat", "lon", "p_value"]]
    p_df = p_df.loc[:, ["lon", "lat", "p_value"]]

    return p_df[p_df["p_value"] < sig_lvl]

In [ ]:
# KS significance test — does pooling change which areas are significant?
with ProgressBar():
    pooled_p_df = get_ks_pval_df(hist_pool, ssp_pool)

In [ ]:
with ProgressBar():
    hist_mmm_rename = box_hist_wrf.mean(
        dim="sim").rename({"time": "index"}).compute()
    ssp_mmm_rename = box_ssp_wrf.mean(dim="sim").rename({
        "time": "index"}).compute()
    mmm_p_df = get_ks_pval_df(hist_mmm_rename, ssp_mmm_rename)

#### Step 3a: Multi-model mean vs. pooled — spatial comparison

Compare the 99th-percentile change maps computed two ways: via the **multi-model mean** (standard approach) and via **pooling** (recommended for high-IV variables). Stippled dots mark grid cells where the KS test finds statistical significance (p < 0.05).

In [ ]:
# Build comparison maps
mmm_diff_plot = make_precip_map(
    delta_mmm,  "Multi-model mean",  None, None, diverging=True, width=300, height=300)
pool_diff_plot = make_precip_map(
    delta_pool, "Multi-model pool",  None, None, diverging=True, width=300, height=300)

mmm_sig = mmm_diff_plot * \
    hv.Points(mmm_p_df).opts(color="k", marker="dot", size=4)
pool_sig = pool_diff_plot * \
    hv.Points(pooled_p_df).opts(color="k", marker="dot", size=4)

comparison = pn.Tabs(
    ("Difference — no stippling",        mmm_diff_plot + pool_diff_plot),
    ("Difference — with KS significance", mmm_sig + pool_sig),
    dynamic=False,
)
comparison

**Reading the maps:**  
- Differences between the mean and pooled maps are subtle in many areas — both approaches capture the broad spatial pattern.  
- However, the **significance pattern differs**: the multi-model mean can show statistical significance even where internal variability is large enough to make that significance spurious (overconfidence from small sample size).  
- The pooled approach inflates the effective sample size, making significance tests more conservative and more honest about what the data can support.

> For variables or regions where internal variability is high (as shown in Step 2b), reporting significance from a multi-model mean **overstates confidence**. Use the pooled result as the primary output.

#### Step 3b: Sample size effect — why pooling matters quantitatively

The cell below prints a concrete sample size comparison between the two approaches, making the statistical argument explicit.

In [ ]:
n_sims = len(box_hist_wrf.simulation.values)
n_months = len(box_hist_wrf.time.values)
n_pool = n_sims * n_months

print("=" * 52)
print("  SAMPLE SIZE: MULTI-MODEL MEAN vs. POOLED")
print("=" * 52)
print(f"  Models available:          {n_sims}")
print(f"  Months per model:          {n_months}")
print()
print(
    f"  MMM sample (per grid cell): {n_months} months × {n_sims} model means")
print(f"  Pooled sample:              {n_pool} months ({n_sims} × {n_months})")
print(f"  Pooling factor:             {n_sims}×")
print("=" * 52)
print()
print("  For a 99th-percentile estimate, a minimum of ~200")
print("  samples is typically needed for statistical stability.")
print(
    f"  Single-model sample: {n_months} → {'sufficient' if n_months >= 200 else 'marginal'}")
print(
    f"  Pooled sample:       {n_pool} → {'sufficient' if n_pool >= 200 else 'marginal'}")

### Step 4: When does internal variability matter?

Internal variability is not equally important for all variables, regions, or time horizons. The guidance below summarises the key rules:

| Context | Internal variability dominates? | Recommended approach |
|---|---|---|
| Near-term (< 20 yr), any variable | **Yes** | Pool ensemble members; acknowledge large uncertainty |
| Long-term (> 40 yr), temperature | **No** — model/scenario dominate | Multi-model ensemble, keep scenarios separate |
| Long-term (> 40 yr), extreme precipitation | **Yes** | Pool; use KS test with conservative thresholds |
| Regional scale | **Yes** — amplified regionally | Pool; avoid single-model conclusions |
| Global scale | **Partial** | Standard multi-model mean is more defensible |

> For a quantitative decomposition of how internal variability compares to model and scenario uncertainty over time, see `uncertainty_decomposition.ipynb`.

**References:**
- [Hawkins & Sutton (2009) — *BAMS*](https://journals.ametsoc.org/view/journals/bams/90/8/2009bams2607_1.xml): the canonical source for the variance decomposition framework used in `uncertainty_decomposition.ipynb`
- [IPCC AR5 uncertainty guidance](https://www.ipcc.ch/site/assets/uploads/2017/08/AR5_Uncertainty_Guidance_Note.pdf): formal IPCC framework for communicating uncertainty
- [climatewest.ca: Uncertainty 101](https://climatewest.ca/2022/09/27/uncertainty-101-understanding-climate-models/): accessible overview of all three uncertainty sources
- [climatedata.ca: Uncertainty in projections](https://climatedata.ca/resource/uncertainty-in-climate-projections/): regional application context